# HW3: Which trucks go to the workshop?

**Name:** _your name_ &nbsp;&nbsp; **Student ID:** _your id_

See `README.md` for the scenario, requirements and grading. This starter loads the data, sets up MLflow, defines the cost, and writes the submission files; the rest is yours. Put your final model and threshold in `hw3_submission.py`.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "2"   # keep OpenMP threads modest on shared machines

import json

import mlflow
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

import hw3_submission as sub

SEED = 0
C_FP, C_FN = 10, 500        # cost of an unnecessary workshop check / of a missed APS failure
EXPERIMENT = "hw3-truck-checks"
mlflow.set_tracking_uri("sqlite:///mlflow.db")   # browse: uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
mlflow.set_experiment(EXPERIMENT)

/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/Users/hemekci/Desktop/Hacettepe/AIE683-Machine-Learning/homeworks/hw3-truck-checks/mlruns/1', creation_time=1790703365442, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790703365442, lifecycle_stage='active', name='hw3-truck-checks', tags={}, trace_location=None, workspace='default'>

In [2]:
aps = fetch_openml(data_id=41138, as_frame=True)
X = aps.data.astype("float32")                     # exactly what the grader passes to your pipeline
y = (aps.target == "pos").astype(int).to_numpy()   # 1 = failure in the air pressure system
print(X.shape, f"failures: {y.sum()} ({y.mean():.2%}); missing cells: {X.isna().mean().mean():.1%}")


def cost_per_truck(y_true, flagged) -> float:
    """Average cost per truck of a flagging decision (the grader uses the same definition)."""
    y_true, flagged = np.asarray(y_true), np.asarray(flagged).astype(bool)
    fp = np.sum(flagged & (y_true == 0))
    fn = np.sum(~flagged & (y_true == 1))
    return (C_FP * fp + C_FN * fn) / len(y_true)

(76000, 170) failures: 1375 (1.81%); missing cells: 8.3%


## 1. Protocol and metric

In [3]:
# TODO

## 2. Candidates under nested cross-validation (one MLflow run per outer fold)

In [4]:
# TODO

## 3. Statistical comparison from the MLflow runs

In [5]:
# TODO

## 4. Threshold and final test

In [6]:
# TODO

## 5. Check the submission module

In [7]:
try:
    model = sub.build_pipeline()
    print("module OK:", type(model).__name__, "threshold", sub.THRESHOLD)
except NotImplementedError as e:
    print("not done yet:", e)

not done yet: TODO: build your pipeline


## 6. Answers and export

A reported number is `{"value": float, "run_id": "<MLflow run id>", "metric": "<metric name in that run>"}`.
Family names: `"logistic_regression"`, `"hist_gradient_boosting"`, `"random_forest"`, `"lightgbm"`, `"xgboost"`, `"other"`.

In [8]:
answers = {
    "student_id": None,                  # str
    "selection_metric": None,            # "accuracy" | "roc_auc" | "average_precision" | "f1" | "expected_cost"
    "cost_never_check": None,            # float: cost per truck if no truck is ever flagged (your test set)
    "cost_at_default_threshold": None,   # reported number: chosen model at threshold 0.5, your test set
    "threshold": None,                   # float: must equal hw3_submission.THRESHOLD
    "test_cost_per_truck": None,         # reported number: chosen model at your threshold, your test set
    "expected_cost_per_truck": None,     # float: the cost you expect on new trucks
    "chosen_model": None,                # family name
    "comparison": {                      # your nested-CV comparison of two families
        "model_a": None, "model_b": None,    # family names
        "metric": None,                      # MLflow metric name logged in every outer-fold run
        "higher_is_better": None,            # bool
        "fold_run_ids_a": None,              # list of run ids, one per outer fold, in fold order
        "fold_run_ids_b": None,              # same folds, same order
        "n_train": None, "n_test": None,     # ints: outer training / test fold sizes
        "p_naive": None,                     # float: paired t-test
        "p_corrected": None,                 # float: corrected resampled t-test (Week 8)
        "better_model": None,                # family name or "no_evidence" (at the 5% level)
    },
    "memo": {
        "decision": None,                    # "deploy_model" | "keep_current_policy"
        "expected_savings_per_truck": None,  # float: cost_never_check - expected_cost_per_truck
        "main_risk": None,                   # "cost_estimates_wrong" | "threshold_drift" | "class_imbalance" |
                                             # "data_quality" | "none"
        "justification": "",                 # at most 150 words for the fleet maintenance manager (not graded)
    },
}
with open("answers.json", "w") as f:
    json.dump(answers, f, indent=2)
mlflow.search_runs(experiment_names=[EXPERIMENT]).to_csv("mlflow_runs.csv", index=False)
print("wrote answers.json and mlflow_runs.csv")

wrote answers.json and mlflow_runs.csv


## AI-use statement

TODO: which AI tools you used, for what, and how you checked their output ("None" is valid, but write a full sentence).